# SupplyMind OpenEnv — TRL + Unsloth DPO Training in Colab

**Meta PyTorch × Scaler OpenEnv Hackathon Finals (2026-04-25/26) — minimum-requirement notebook**

This notebook satisfies the hackathon's hard gate: *"Show a minimal training script for your environment using Unsloth or HF TRL in Colab"* — using **both** (self-serve guide §10: *"TRL for RL training algorithms, Unsloth to make RL training and inference more efficient, OpenEnv to standardize environment interaction"*).

What it does, end-to-end on a free Colab T4:
1. Clones the SupplyMind repo (the OpenEnv-compliant supply-chain risk env).
2. Loads **21 real preference pairs** generated by a 3-judge LLM panel (Qwen-2.5-14B × Mistral-Nemo × DeepSeek-R1) over real 2024-2026 crisis scenarios.
3. Loads **Qwen-2.5-0.5B-Instruct** via **Unsloth `FastLanguageModel`** (4-bit NF4, 2-5× faster + 70% less VRAM than vanilla TRL). Falls back cleanly to vanilla `transformers` if Unsloth isn't available.
4. Fine-tunes with **TRL `DPOTrainer`** + LoRA (r=8) — the judge-model that scores agent actions inside the OpenEnv `/step` loop.
5. Plots the training loss curve + chosen/rejected reward margins.
6. Compares base vs DPO-tuned outputs on a held-out real event (2026 Gulf of Oman cargo-ship seizure).
7. Shows how the tuned judge plugs back into the live HF-Space OpenEnv at https://huggingface.co/spaces/Shaurya-Noodle/Supplymind

Runtime: ~5-10 min on Colab free T4 with Unsloth (vs 12-18 min on vanilla). No API keys, no external services.

**Why DPO here, then GRPO onsite?**
Self-serve guide §3: *"a little SFT first, then RL."* We have no curated SFT data but we do have 21 real preference pairs from a 3-judge panel — DPO is the natural warm-start. The full **env-connected GRPO** stage (§11 RLVR) lives at [`versions/v5_phoenix/roll_integration/dpo_judge/train_grpo_live_env.py`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/versions/v5_phoenix/roll_integration/dpo_judge/train_grpo_live_env.py) and sends **live HTTP POST /analyst/grade** on the running env for every reward signal — no static dataset anywhere in the reward path.

## 1. Environment setup

Pinned versions — this combination was regression-tested locally to avoid the `peft>=0.19 × torch<2.6` skew that breaks DPO.

In [ ]:
# Unsloth is the hackathon guide's §10 preferred stack companion to TRL — 2-5x
# faster + 70% less VRAM. We install it and fall back to vanilla transformers
# if it's unavailable on the runtime.
!pip install -q \
    "unsloth @ git+https://github.com/unslothai/unsloth.git" \
    "trl==0.12.2" \
    "transformers==4.46.3" \
    "peft>=0.12,<0.15" \
    "accelerate>=0.30,<1.0" \
    "datasets>=2.16" \
    "bitsandbytes>=0.43" \
    matplotlib

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Enable GPU via Runtime > Change runtime type > T4 GPU.")

try:
    import unsloth  # noqa: F401
    USE_UNSLOTH = True
    print(f"Unsloth  : {unsloth.__version__} (primary path — 2-5x faster TRL)")
except Exception as _e:
    USE_UNSLOTH = False
    print(f"Unsloth  : unavailable ({_e}); falling back to vanilla transformers")


## 2. Clone the SupplyMind OpenEnv repo

The OpenEnv compliance (`/reset`, `/step`, `/state` FastAPI + Pydantic Action/Observation schemas) lives at [server/app.py](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/server/app.py) with the OpenEnv adapter at [server/openenv_adapter.py](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/server/openenv_adapter.py). We only need the preference-pair JSONL here.

In [ ]:
import os, json
from pathlib import Path

if not Path("Sleep-Token").exists():
    !git clone --depth 1 https://github.com/ShAuRyA-Noodle/Sleep-Token.git
os.chdir("Sleep-Token")

DATA = Path("versions/v5_phoenix/roll_integration/dpo_judge/data/preference_pairs.jsonl")
assert DATA.exists(), DATA

pairs = [json.loads(l) for l in DATA.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"Loaded {len(pairs)} preference pairs")
print(f"Sample scenario: {pairs[0]['meta']['scenario_id']}")
print(f"Ground-truth risk: {pairs[0]['meta']['gt_risk']}")
print(f"Quality gap (chosen - rejected): {pairs[0]['meta']['quality_gap']}")

## 3. Build the HuggingFace Dataset

Hold out 2 pairs for before/after eval; train on the other 19. Tiny dataset → perfect for a reproducible Colab demo.

In [ ]:
from datasets import Dataset
import random

random.seed(42)
random.shuffle(pairs)
eval_pairs = pairs[:2]
train_pairs = pairs[2:]

train_ds = Dataset.from_list([
    {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
    for p in train_pairs
])
print(f"Train: {len(train_ds)} | Eval: {len(eval_pairs)}")
print(f"\nEval held-out scenarios:")
for p in eval_pairs:
    print(f"  - {p['meta']['scenario_id']} (gt={p['meta']['gt_risk']})")

## 4. Load Qwen-2.5-0.5B-Instruct + LoRA adapter

Primary path: **Unsloth `FastLanguageModel`** with 4-bit NF4 quantisation — hackathon guide §10 (TRL + **Unsloth** + OpenEnv). Falls back cleanly to vanilla transformers if Unsloth isn't available. LoRA on all projection layers.

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN = 2048
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    policy, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,                # auto-pick (bf16 on A100, fp16 on T4)
        load_in_4bit=True,         # NF4 — 70% VRAM reduction
    )
    policy = FastLanguageModel.get_peft_model(
        policy,
        r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=LORA_TARGETS,
        bias="none",
        use_gradient_checkpointing="unsloth",   # Unsloth's custom checkpointing
        random_state=42,
    )
    # Unsloth builds its own PEFT wrapper; pass None to DPOTrainer(peft_config=...)
    peft_cfg_for_trl = None
    print("Loaded via Unsloth FastLanguageModel + get_peft_model (4-bit NF4)")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig
    tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    policy = AutoModelForCausalLM.from_pretrained(
        BASE, torch_dtype=dtype, trust_remote_code=True,
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    peft_cfg_for_trl = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=LORA_TARGETS,
    )
    print(f"Loaded via vanilla transformers ({dtype})")

tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print(f"Base model: {BASE}")
print(f"LoRA: r=8, alpha=16, 7 target modules")

## 5. Snapshot base-model output *before* training

We'll compare this against the DPO-tuned output after Section 7.

In [ ]:
def generate(model, prompt, max_new_tokens=220):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1400).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

eval_prompt = eval_pairs[0]["prompt"]
eval_gt = eval_pairs[0]["meta"]["gt_risk"]

baseline_out = generate(policy, eval_prompt)
print(f"BASE model on: {eval_pairs[0]['meta']['scenario_id']}")
print(f"Ground truth : {eval_gt}")
print("---")
print(baseline_out[:500])

## 6. Train with TRL `DPOTrainer`

Direct Preference Optimization — no reward model needed, and `ref_model=None` lets TRL use the adapter-disabled policy as reference (standard PEFT-DPO pattern). Logs loss every step.

In [ ]:
from trl import DPOTrainer, DPOConfig

cfg = DPOConfig(
    output_dir="./dpo_out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False if USE_UNSLOTH else True,   # Unsloth handles GC internally
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    beta=0.1,
    max_length=2048,
    max_prompt_length=1024,
    report_to=[],
    eval_strategy="no",
    do_eval=False,
    remove_unused_columns=False,
    generate_during_eval=False,
)

import time as _t
_t0 = _t.time()
trainer = DPOTrainer(
    model=policy,
    ref_model=None,
    args=cfg,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    peft_config=peft_cfg_for_trl,    # None for Unsloth, LoraConfig for vanilla
)
trainer.train()
print(f"\nTraining complete in {_t.time() - _t0:.1f}s"
      f" (Unsloth: {USE_UNSLOTH}).")

## 7. Plot the loss curve

Judges want observable evidence of training progress. This is it — saved as `training_loss.png`.

In [ ]:
import matplotlib.pyplot as plt

log = [e for e in trainer.state.log_history if "loss" in e]
steps = [e["step"] for e in log]
losses = [e["loss"] for e in log]
rewards_chosen = [e.get("rewards/chosen") for e in log if e.get("rewards/chosen") is not None]
rewards_rejected = [e.get("rewards/rejected") for e in log if e.get("rewards/rejected") is not None]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(steps, losses, marker="o", color="#c44", lw=1.2)
axes[0].set_xlabel("Step"); axes[0].set_ylabel("DPO loss")
axes[0].set_title("SupplyMind DPO training loss")
axes[0].grid(alpha=0.3)

if rewards_chosen and rewards_rejected:
    rc_steps = [e["step"] for e in log if e.get("rewards/chosen") is not None]
    axes[1].plot(rc_steps, rewards_chosen, marker="o", color="#48a", lw=1.2, label="chosen")
    axes[1].plot(rc_steps, rewards_rejected, marker="x", color="#c73", lw=1.2, label="rejected")
    axes[1].set_xlabel("Step"); axes[1].set_ylabel("Implicit reward (log-prob ratio × β)")
    axes[1].set_title("Chosen vs Rejected reward margin (should diverge)")
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_loss.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Final loss: {losses[-1]:.4f} (from {losses[0]:.4f} at step 1)")

## 8. After-training inference — compare against baseline

Same held-out scenario as Section 5. The DPO-tuned model should output a response closer to the **chosen** preference (ground-truth risk level + calibrated confidence).

In [ ]:
trained_out = generate(trainer.model, eval_prompt)

print(f"Scenario: {eval_pairs[0]['meta']['scenario_id']}")
print(f"Ground truth risk: {eval_gt}\n")
print("BEFORE training (base model):")
print("-" * 50)
print(baseline_out[:400])
print("\nAFTER DPO training:")
print("-" * 50)
print(trained_out[:400])

def extract_risk(s):
    for r in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if r in s.upper(): return r
    return "UNKNOWN"

print(f"\nRisk extracted — base: {extract_risk(baseline_out)}  |  trained: {extract_risk(trained_out)}  |  gt: {eval_gt}")

## 9. How the tuned judge plugs into the live OpenEnv

The adapter produced above is the same LoRA used by the SupplyMind judge inside `/step`:
```
observation ──► LLM agent ──► action ──► env /step ──► judge(LoRA) ──► reward ──► OpenEnv response
```

Live example against the deployed HF Space (no local GPU needed):

In [ ]:
import urllib.request, urllib.error, json as _json
SPACE = "https://shaurya-noodle-supplymind.hf.space"

for ep in ["/health", "/tasks", "/phoenix/status"]:
    try:
        with urllib.request.urlopen(SPACE + ep, timeout=15) as r:
            print(f"GET {ep} -> {r.status}")
            print(r.read()[:180].decode(errors='ignore'), "...")
    except urllib.error.URLError as e:
        print(f"GET {ep} -> {e}")
    print()

## 10. Next steps

- **Env-connected GRPO (RLVR)** — [`versions/v5_phoenix/roll_integration/dpo_judge/train_grpo_live_env.py`](https://github.com/ShAuRyA-Noodle/Sleep-Token/blob/main/versions/v5_phoenix/roll_integration/dpo_judge/train_grpo_live_env.py). Every reward is an HTTP `POST /analyst/grade` on the running env — truly env-connected per self-serve guide §11. Three independent reward components (match 0.7 + format 0.2 + length 0.1) hardened against reward hacking per §8. Dry-run proven (`correct=0.900`, `wrong=0.200`, `gap=0.700`).
- **Autoresearch loop** — `versions/v4_arcadia_live/autoresearch/` runs Karpathy-style candidate-training experiments with bootstrap CI95 accept/reject. Already achieved +0.148 CI95 lift over baseline on the RL policy (R6 Euclidian).
- **Live demo** — https://huggingface.co/spaces/Shaurya-Noodle/Supplymind · POST `/v3/e2e` with any scenario and observe the input-dependent risk level + R5 RAG retrieval + R6-anchored conformal forecast + real `supplymind_env.reset` observation feeding the MaskablePPO ONNX policy.

*Built by [@ShAuRyA-Noodle](https://github.com/ShAuRyA-Noodle) for the Meta PyTorch × Scaler OpenEnv Hackathon Finals 2026. No synthetic data in the reward path; every stage tagged with `inference_type` provenance.*